# SYNCHAIN AI — Inference Notebook
**Team:** Ctrl-Alt-Win | Universitas Telkom  
**Version:** 2.0  
**Track:** C — The Explainable Oracle (Predictive Analytics)

---
Notebook ini mencakup:
1. Load model yang sudah ditraining
2. Load test data dari folder `test_data/`
3. Inferensi prediksi demand
4. Explainability per prediksi (SHAP + human-readable)
5. Restock recommendation
6. Export hasil inferensi

> ⚠️ **Constraint Check:** Seluruh inferensi berjalan secara **offline/localhost**. Tidak ada API eksternal yang dipanggil.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import json
import time
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
from synchain_ai import SynChainAI_Advanced

print('✅ Library berhasil diimport')
print(f'📅 Waktu inferensi: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## 2. Load Model

In [ ]:
ai = SynChainAI_Advanced(model_path='synchain_model.pkl')
ai.load_model()

print(f'\n📦 Info Model:')
print(f'   Versi   : {ai.metadata["version"]}')
print(f'   Tim     : {ai.metadata["team"]}')
print(f'   Dibuat  : {ai.metadata["created"]}')
print(f'   Fitur   : {len(ai.feature_columns)} kolom')

## 3. Load Test Data
> **Constraint Check:** Data test diambil dari folder `test_data/` yang **terpisah** dari `train_data/` — pemisahan dilakukan sebelum preprocessing.

In [ ]:
print('📂 Memuat data dari folder test_data/ ...')
test_files = [
    'test_data/retail_demand_dataset.csv',
]

test_dfs = []
for path in test_files:
    try:
        df_part = ai.load_local_data(path)
        test_dfs.append(df_part)
        print(f'  ✅ {path} — {len(df_part)} baris')
    except FileNotFoundError:
        print(f'  ⚠️ File tidak ditemukan: {path} (skip)')

if test_dfs:
    df_test = pd.concat(test_dfs, ignore_index=True)
    print(f'\n📊 Total test data: {df_test.shape[0]} baris, {df_test.shape[1]} kolom')
else:
    # Fallback: buat dummy test data untuk demo
    print('\n⚠️  Tidak ada file test ditemukan. Menggunakan dummy data untuk demo.')
    np.random.seed(99)
    df_test = pd.DataFrame({
        'product_id': np.random.randint(1, 50, 20),
        'category': np.random.choice(['Electronics', 'Clothing', 'Food'], 20),
        'price': np.random.uniform(10, 1000, 20).round(2),
        'month': np.random.randint(1, 13, 20),
        'supplier': np.random.choice(['A', 'B', 'C'], 20),
    })
    print(f'   Dummy data: {df_test.shape[0]} baris')

df_test.head()

## 4. Preprocessing Test Data

In [ ]:
def preprocess_test(df_raw, ai_instance):
    """Preprocessing test data menggunakan encoder & scaler yang sudah difit saat training."""
    data = df_raw.copy()
    
    # Drop missing values
    data = data.dropna()
    
    # Date feature engineering
    date_cols = data.select_dtypes(include=['datetime64']).columns
    for col in date_cols:
        data[f'{col}_year']      = data[col].dt.year
        data[f'{col}_month']     = data[col].dt.month
        data[f'{col}_day']       = data[col].dt.day
        data[f'{col}_dayofweek'] = data[col].dt.dayofweek
        data[f'{col}_quarter']   = data[col].dt.quarter
        data = data.drop(col, axis=1)
    
    # Encode menggunakan label encoder dari training
    for col, le in ai_instance.label_encoders.items():
        if col in data.columns:
            data[col] = data[col].astype(str).apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else le.transform([le.classes_[0]])[0]
            )
    
    # Pastikan kolom sesuai dengan saat training
    for col in ai_instance.feature_columns:
        if col not in data.columns:
            data[col] = 0
    data = data[ai_instance.feature_columns]
    
    # Scale menggunakan scaler dari training (bukan fit ulang!)
    X_scaled = ai_instance.scaler.transform(data)
    return pd.DataFrame(X_scaled, columns=ai_instance.feature_columns)

X_test = preprocess_test(df_test, ai)
print(f'✅ Preprocessing test data selesai: {X_test.shape[0]} sampel, {X_test.shape[1]} fitur')

## 5. Inferensi Batch
> Mengukur waktu inferensi per sampel untuk memastikan memenuhi constraint kecepatan.

In [ ]:
print('⏱️  Mengukur kecepatan inferensi...')

# Batch inference
t_start = time.time()
predictions = ai.model.predict(X_test)
t_end = time.time()

total_time = t_end - t_start
time_per_sample = total_time / len(X_test) if len(X_test) > 0 else 0

print(f'\n📊 Hasil Batch Inferensi:')
print(f'   Jumlah sampel     : {len(X_test)}')
print(f'   Total waktu       : {total_time:.4f} detik')
print(f'   Waktu per sampel  : {time_per_sample:.4f} detik')
print(f'   Limit constraint  : 3 detik per sampel')
status = '✅ LULUS' if time_per_sample < 3.0 else '❌ MELEBIHI BATAS'
print(f'   Status            : {status}')

## 6. Explainability per Prediksi (SHAP)
> **Constraint Check (Track C):** Output wajib menyebutkan ≥3 variabel teratas beserta arah pengaruhnya (positif/negatif).

In [ ]:
print('🔍 Menghitung SHAP values untuk test data...')
explainer = shap.TreeExplainer(ai.model)
shap_values_test = explainer.shap_values(X_test)
print(f'✅ SHAP values selesai untuk {len(X_test)} sampel')

In [ ]:
def generate_human_readable_explanation(shap_vals, feature_names, prediction, base_value, top_n=3):
    """
    Menghasilkan penjelasan prediksi yang dapat dibaca manusia.
    Constraint Track C: wajib menyebutkan >= 3 variabel teratas beserta arah pengaruh.
    """
    shap_pairs = list(zip(feature_names, shap_vals))
    shap_sorted = sorted(shap_pairs, key=lambda x: abs(x[1]), reverse=True)[:top_n]
    
    lines = []
    lines.append(f'📌 Prediksi demand: {prediction:.2f} unit')
    lines.append(f'📍 Baseline rata-rata: {base_value:.2f} unit')
    lines.append(f'\n🔍 {top_n} Faktor Utama yang Mempengaruhi Prediksi:')
    
    for i, (feat, val) in enumerate(shap_sorted, 1):
        direction = '⬆️  MENINGKATKAN' if val > 0 else '⬇️  MENURUNKAN'
        lines.append(f'   {i}. [{direction}] {feat}: kontribusi {val:+.4f}')
    
    return '\n'.join(lines)


# Tampilkan penjelasan untuk 3 sampel pertama
n_display = min(3, len(X_test))
print(f'📋 Penjelasan Prediksi (menampilkan {n_display} sampel):')
print('=' * 60)

for idx in range(n_display):
    explanation = generate_human_readable_explanation(
        shap_values_test[idx],
        ai.feature_columns,
        predictions[idx],
        explainer.expected_value
    )
    print(f'\n🧪 Sampel #{idx}:')
    print(explanation)
    print('-' * 60)

In [ ]:
# SHAP Waterfall plot untuk sampel pertama
import os
os.makedirs('plots', exist_ok=True)

shap_exp = shap.Explanation(
    values=shap_values_test[0],
    base_values=explainer.expected_value,
    data=X_test.iloc[0].values,
    feature_names=ai.feature_columns
)
plt.figure()
shap.plots.waterfall(shap_exp, show=False)
plt.title('SHAP Waterfall — Inferensi Sampel #0', fontsize=12)
plt.tight_layout()
plt.savefig('plots/inference_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP waterfall inferensi disimpan')

## 7. Prediksi dengan Confidence Interval

In [ ]:
# Contoh prediksi single input dengan confidence interval
sample_input = df_test.iloc[0].to_dict()
print(f'📥 Input Sampel: {sample_input}')

result = ai.get_predictions_with_confidence(sample_input, n_iterations=20)

print(f'\n📊 Hasil Prediksi dengan Confidence Interval:')
print(f'   Prediksi (mean) : {result["prediction"]:.2f} unit')
print(f'   Std Deviasi     : {result["std"]:.2f}')
print(f'   CI {result["confidence"]}%          : [{result["ci_lower"]:.2f}, {result["ci_upper"]:.2f}]')

## 8. Restock Recommendation

In [ ]:
print('📦 Restock Recommendation untuk seluruh test data:')
print('=' * 60)

restock_results = []
for idx, pred in enumerate(predictions):
    current_stock = np.random.randint(20, 200)  # Ganti dengan data stok aktual jika tersedia
    rec = ai.generate_restock_recommendation(
        current_stock=current_stock,
        predicted_demand=float(pred),
        lead_time=7,
        safety_margin=0.2
    )
    rec['sample_id'] = idx
    restock_results.append(rec)

restock_df = pd.DataFrame(restock_results)
print(restock_df[['sample_id', 'action', 'priority', 'current_stock',
                   'predicted_demand', 'restock_quantity']].head(10).to_string(index=False))

# Ringkasan
print(f'\n📋 Ringkasan Restock:')
print(restock_df['action'].value_counts().to_string())

## 9. Visualisasi Hasil Inferensi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribusi prediksi
axes[0].hist(predictions, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Hasil Prediksi Demand')
axes[0].set_xlabel('Predicted Demand (unit)')
axes[0].set_ylabel('Frekuensi')

# Pie chart status restock
action_counts = restock_df['action'].value_counts()
colors = ['#e74c3c' if 'RESTOCK' in a else '#2ecc71' for a in action_counts.index]
axes[1].pie(action_counts.values, labels=action_counts.index,
            colors=colors, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Status Restock Rekomendasi')

plt.suptitle('SynChain AI — Hasil Inferensi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/inference_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot ringkasan inferensi disimpan')

## 10. Export Hasil Inferensi

In [ ]:
# Susun dataframe hasil
results_df = df_test.copy().reset_index(drop=True)
results_df['predicted_demand'] = predictions
results_df['ci_lower'] = results_df['predicted_demand'] * 0.95
results_df['ci_upper'] = results_df['predicted_demand'] * 1.05
results_df['restock_action'] = restock_df['action'].values
results_df['restock_priority'] = restock_df['priority'].values
results_df['restock_qty'] = restock_df['restock_quantity'].values

# Top SHAP feature per baris
top1_features, top2_features, top3_features = [], [], []
top1_dirs, top2_dirs, top3_dirs = [], [], []

for idx in range(len(X_test)):
    shap_pairs = sorted(zip(ai.feature_columns, shap_values_test[idx]),
                        key=lambda x: abs(x[1]), reverse=True)
    for i, (lst_feat, lst_dir) in enumerate([
        (top1_features, top1_dirs),
        (top2_features, top2_dirs),
        (top3_features, top3_dirs)
    ]):
        feat, val = shap_pairs[i]
        lst_feat.append(feat)
        lst_dir.append('positif' if val > 0 else 'negatif')

results_df['top1_feature'] = top1_features
results_df['top1_direction'] = top1_dirs
results_df['top2_feature'] = top2_features
results_df['top2_direction'] = top2_dirs
results_df['top3_feature'] = top3_features
results_df['top3_direction'] = top3_dirs

# Simpan ke CSV
output_path = 'synchain_inference_results.csv'
results_df.to_csv(output_path, index=False)
print(f'✅ Hasil inferensi tersimpan: {output_path}')
results_df.head()

In [ ]:
print('=' * 58)
print('✅  INFERENSI SELESAI — SYNCHAIN AI v2.0')
print('=' * 58)
print(f'  Total sampel diproses : {len(X_test)}')
print(f'  Waktu per sampel      : {time_per_sample:.4f} detik')
print(f'  Constraint 3 detik    : {status}')
print(f'  SHAP explainability   : ✅ aktif')
print(f'  Human-readable output : ✅ aktif (top 3 fitur per prediksi)')
print(f'  Offline / localhost   : ✅ terpenuhi')
print(f'  Output file           : synchain_inference_results.csv')
print('=' * 58)